# Results for model: institute-of-science-tokyo/llama-3.1-swallow-8b-instruct-v0.1

In [1]:
import polars as pl
import xgboost as xgb

# Load data
df = pl.read_parquet('./jane_street_data/train.parquet')

# Feature Engineering
df = df.with_column(
    pl.when(df['responder_6'] > df['feature_00']).then(1).otherwise(0).rolling('date_id', min_periods=1, center=True, window=10).apply(lambda x: x.quantile(0.85)).alias('top_quantile')
)

# Train XGBoost Regressor
train_df = df.filter(pl.col('date_id').is_not_null())
test_df = df.filter(pl.col('date_id').is_null())

xgb_model = xgb.XGBRegressor(objective='reg:squarederror')
xgb_model.fit(train_df.drop('responder_6').values, train_df['responder_6'].values)

# Make predictions
predictions = xgb_model.predict(test_df.drop('responder_6').values)

# Output predictions
test_df = test_df.with_column(pl.Series(predictions, name='predictions'))
test_df.write_parquet('./jane_street_data/predictions.parquet')

FileNotFoundError: The system cannot find the path specified. (os error 3): ./jane_street_data/train.parquet